<a href="https://colab.research.google.com/github/Jassmine11/cfpb-complaints-project/blob/main/Copy_of_Experiment_With_Various_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd

DRIVE_BASE = Path("/content/drive/MyDrive/capstone_artifacts")
CLEANED_PATH = DRIVE_BASE / "data" / "cleaned" / "cfpb_cleaned_step5.csv"
assert CLEANED_PATH.exists(), f"Cleaned file not found: {CLEANED_PATH}"
df = pd.read_csv(CLEANED_PATH)
print("Loaded cleaned file, shape:", df.shape)
df.head(2)


Loaded cleaned file, shape: (26861, 17)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company,State,ZIP code,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,text_len_words,is_short,is_long
0,2019-08-29,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,Their investigation did not fix an error on yo...,"On XX/XX/XXXX, I received a personal loan from...","MARINER FINANCE, LLC",MS,39702,Web,2019-08-29,Closed with explanation,Yes,3358661,404,False,False
1,2025-07-22,Credit card,Store credit card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,My line of credit was used by an identity thie...,"CITIBANK, N.A.",IA,52402,Web,2025-07-22,Closed with non-monetary relief,Yes,14815846,213,False,False


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

TEXT_COL = "Consumer complaint narrative"
TARGET_COL = "Product"

# Optionally collapse rare labels (same logic you used before)
min_count = 20
vc = df[TARGET_COL].value_counts()
rare = vc[vc < min_count].index.tolist()
df[TARGET_COL] = df[TARGET_COL].apply(lambda x: "Other" if x in rare else x)

le = LabelEncoder()
df["y"] = le.fit_transform(df[TARGET_COL])

X = df[TEXT_COL].values
y = df["y"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    stratify=y, random_state=42)

print("Train size:", len(X_train), "Test size:", len(X_test), "Classes:", len(le.classes_))


Train size: 21488 Test size: 5373 Classes: 18


In [ ]:
SPLITS_DIR = DRIVE_BASE / "artifacts" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump((X_train, X_test, y_train, y_test, le), SPLITS_DIR / "split_and_le_step7.joblib")
print("Saved splits & label encoder to:", SPLITS_DIR / "split_and_le_step7.joblib")


Saved splits & label encoder to: /content/drive/MyDrive/capstone_artifacts/artifacts/splits/split_and_le_step7.joblib


In [ ]:
!ls -la "/content/drive/MyDrive/capstone_artifacts/data/cleaned"
!ls -la "/content/drive/MyDrive/capstone_artifacts/artifacts/splits"


total 71323
-rw------- 1 root root 36518027 Apr 19 19:49 cfpb_cleaned_step5.csv
-rw------- 1 root root      278 Apr 19 19:49 cfpb_cleaned_step5_provenance.json
-rw------- 1 root root 36515421 Apr 19 19:49 cfpb_cleaned_step5_redacted.csv
total 27267
-rw------- 1 root root 27920705 Apr 19 19:53 split_and_le_step7.joblib


In [ ]:
# Quick baseline: TF-IDF + LogisticRegression using saved splits
import joblib, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DRIVE_BASE = Path("/content/drive/MyDrive/capstone_artifacts")
SPLITS_PATH = DRIVE_BASE / "artifacts" / "splits" / "split_and_le_step7.joblib"
OUT_DIR = Path("step7_results"); OUT_DIR.mkdir(exist_ok=True)
RES_DRIVE = DRIVE_BASE / "results"; RES_DRIVE.mkdir(parents=True, exist_ok=True)

# load splits + label encoder
X_train, X_test, y_train, y_test, le = joblib.load(SPLITS_PATH)
print("Loaded splits — train:", len(X_train), "test:", len(X_test), "classes:", len(le.classes_))

# build and train baseline
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words="english")
model = Pipeline([("tfidf", tfidf), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1, random_state=42))])

t0 = time.time()
model.fit(X_train, y_train)
train_time = time.time() - t0

# evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
wf1 = f1_score(y_test, y_pred, average="weighted")
mf1 = f1_score(y_test, y_pred, average="macro")
report = classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print(f"Baseline TF-IDF+LogReg — acc: {acc:.4f}, weighted_f1: {wf1:.4f}, macro_f1: {mf1:.4f}, train_s: {train_time:.1f}")
print("\nClassification report:\n", report)

# save artifacts locally and to Drive
joblib.dump(model, OUT_DIR/"tfidf_logreg_baseline_model.joblib")
with open(OUT_DIR/"tfidf_logreg_baseline_report.txt","w") as f: f.write(report)
json.dump({"accuracy":acc,"weighted_f1":wf1,"macro_f1":mf1,"train_time_s":train_time}, open(OUT_DIR/"tfidf_logreg_baseline_metrics.json","w"))
np.save(OUT_DIR/"tfidf_logreg_baseline_confusion.npy", cm)

# copy summary artifacts to Drive results folder
shutil.copy2(OUT_DIR/"tfidf_logreg_baseline_model.joblib", RES_DRIVE/"tfidf_logreg_baseline_model.joblib")
shutil.copy2(OUT_DIR/"tfidf_logreg_baseline_report.txt", RES_DRIVE/"tfidf_logreg_baseline_report.txt")
shutil.copy2(OUT_DIR/"tfidf_logreg_baseline_metrics.json", RES_DRIVE/"tfidf_logreg_baseline_metrics.json")
shutil.copy2(OUT_DIR/"tfidf_logreg_baseline_confusion.npy", RES_DRIVE/"tfidf_logreg_baseline_confusion.npy")

print("Saved baseline artifacts to:", OUT_DIR, "and copied to Drive:", RES_DRIVE)


Loaded splits — train: 21488 test: 5373 classes: 18
Baseline TF-IDF+LogReg — acc: 0.7065, weighted_f1: 0.7137, macro_f1: 0.4735, train_s: 39.6

Classification report:
                                                                               precision    recall  f1-score   support

                                                     Bank account or service       0.39      0.38      0.39        29
                                                 Checking or savings account       0.71      0.77      0.73       264
                                                               Consumer Loan       0.25      0.07      0.11        15
                                                                 Credit card       0.45      0.54      0.49       176
                                                 Credit card or prepaid card       0.48      0.59      0.53       185
                                                            Credit reporting       0.18      0.61      0.27        51
     

In [ ]:
# Compute sentence-transformer embeddings, save locally and to Drive, then train LogReg + LightGBM
from sentence_transformers import SentenceTransformer
import numpy as np, joblib, time, shutil
from pathlib import Path
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DRIVE_BASE = Path("/content/drive/MyDrive/capstone_artifacts")
EMB_DIR = DRIVE_BASE / "artifacts" / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("step7_results")

# Load splits
SPLITS_PATH = DRIVE_BASE / "artifacts" / "splits" / "split_and_le_step7.joblib"
X_train, X_test, y_train, y_test, le = joblib.load(SPLITS_PATH)

# Compute embeddings
emb_model = SentenceTransformer("all-MiniLM-L6-v2")
emb_train = emb_model.encode(list(X_train), show_progress_bar=True, batch_size=64)
emb_test = emb_model.encode(list(X_test), show_progress_bar=True, batch_size=64)

# Save embeddings locally and to Drive
np.save(OUT_DIR/"emb_train.npy", emb_train); np.save(OUT_DIR/"emb_test.npy", emb_test)
np.save(EMB_DIR/"emb_train.npy", emb_train); np.save(EMB_DIR/"emb_test.npy", emb_test)
print("Saved embeddings to", OUT_DIR, "and", EMB_DIR)

# Train LogisticRegression on embeddings
t0 = time.time()
clf_emb = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1, random_state=42)
clf_emb.fit(emb_train, y_train)
lr_time = time.time() - t0
y_pred = clf_emb.predict(emb_test)
print("Emb-LogReg — acc:", accuracy_score(y_test, y_pred), "weighted_f1:", f1_score(y_test, y_pred, average='weighted'))
joblib.dump(clf_emb, OUT_DIR/"emb_logreg_model.joblib")
shutil.copy2(OUT_DIR/"emb_logreg_model.joblib", DRIVE_BASE/"results"/"emb_logreg_model.joblib")

# Train LightGBM on embeddings
dtrain = lgb.Dataset(emb_train, label=y_train)
params = {"objective":"multiclass","num_class":len(le.classes_),"metric":"multi_logloss","learning_rate":0.05,"num_leaves":31,"seed":42}
bst = lgb.train(params, dtrain, num_boost_round=200)
class LGBWrap:
    def __init__(self,bst): self.bst=bst
    def predict(self, X): return np.argmax(self.bst.predict(X), axis=1)
lgbw = LGBWrap(bst)
y_pred_lgb = lgbw.predict(emb_test)
print("Emb-LightGBM — acc:", accuracy_score(y_test, y_pred_lgb), "weighted_f1:", f1_score(y_test, y_pred_lgb, average='weighted'))
joblib.dump(bst, OUT_DIR/"emb_lightgbm_bst.pkl")
shutil.copy2(OUT_DIR/"emb_lightgbm_bst.pkl", DRIVE_BASE/"results"/"emb_lightgbm_bst.pkl")

# Save reports/metrics
from sklearn.metrics import classification_report
rep_lr = classification_report(y_test, clf_emb.predict(emb_test), target_names=le.classes_, zero_division=0)
rep_lgb = classification_report(y_test, y_pred_lgb, target_names=le.classes_, zero_division=0)
open(OUT_DIR/"emb_logreg_report.txt","w").write(rep_lr)
open(OUT_DIR/"emb_lightgbm_report.txt","w").write(rep_lgb)
shutil.copy2(OUT_DIR/"emb_logreg_report.txt", DRIVE_BASE/"results"/"emb_logreg_report.txt")
shutil.copy2(OUT_DIR/"emb_lightgbm_report.txt", DRIVE_BASE/"results"/"emb_lightgbm_report.txt")

print("Embedding models trained and artifacts saved.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/336 [00:00<?, ?it/s]

Batches:   0%|          | 0/84 [00:00<?, ?it/s]

Saved embeddings to step7_results and /content/drive/MyDrive/capstone_artifacts/artifacts/embeddings
Emb-LogReg — acc: 0.5931509398846082 weighted_f1: 0.6216221174099347
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.132121 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97920
[LightGBM] [Info] Number of data points in the train set: 21488, number of used features: 384
[LightGBM] [Info] Start training from score -5.239051
[LightGBM] [Info] Start training from score -3.011114
[LightGBM] [Info] Start training from score -5.880905
[LightGBM] [Info] Start training from score -3.418472
[LightGBM] [Info] Start training from score -3.371306
[LightGBM] [Info] Start training from score -4.652240
[LightGBM] [Info] Start training from score -0.904631
[LightGBM] [Info] Start training from score -1.458457
[LightGBM] [Info] Start training from score -2.269987
[LightGBM] [Info] Start training from score -6.884207

In [ ]:
from pathlib import Path
DRIVE_BASE = Path("/content/drive/MyDrive/capstone_artifacts")
print("Cleaned folder listing:")
print(list((DRIVE_BASE/"data"/"cleaned").glob("*"))[:50])



Cleaned folder listing:
[PosixPath('/content/drive/MyDrive/capstone_artifacts/data/cleaned/cfpb_cleaned_step5.csv'), PosixPath('/content/drive/MyDrive/capstone_artifacts/data/cleaned/cfpb_cleaned_step5_redacted.csv'), PosixPath('/content/drive/MyDrive/capstone_artifacts/data/cleaned/cfpb_cleaned_step5_provenance.json')]


In [ ]:
from pathlib import Path
import shutil

DRIVE_BASE = Path("/content/drive/MyDrive/capstone_artifacts")
LOCAL_STEP7 = Path("step7_results")
DRIVE_RESULTS = DRIVE_BASE / "results"
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

if LOCAL_STEP7.exists():
    shutil.copytree(LOCAL_STEP7, DRIVE_RESULTS / "step7_results", dirs_exist_ok=True)
    print("Copied local step7_results ->", DRIVE_RESULTS / "step7_results")
else:
    print("No local step7_results folder found.")


Copied local step7_results -> /content/drive/MyDrive/capstone_artifacts/results/step7_results


In [ ]:
# confirm
!ls -la "/content/drive/MyDrive/capstone_artifacts/data/cleaned"


total 71323
-rw------- 1 root root 36518027 Apr 19 19:49 cfpb_cleaned_step5.csv
-rw------- 1 root root      278 Apr 19 19:49 cfpb_cleaned_step5_provenance.json
-rw------- 1 root root 36515421 Apr 19 19:49 cfpb_cleaned_step5_redacted.csv
